# Exp. 8 – Fundație comună: verificări (S8)

Verifică, fără simulare nouă, că `src/scores_io.py`, `src/alert_budget.py`, `src/report.py` și `src/cost_ledger.py` reproduc cifrele salvate în Exp. 2, 3 și 5, apoi produce tabelul de referință „buget de alarme” pe **validation**. Testul nu este folosit pentru nicio alegere: pe test se citesc doar scorurile deja salvate de Exp. 5, pentru reproducere.

In [ ]:
# Config
SEEDS = [0, 1, 2, 3, 4]
NOISE_LEVELS = [0.0, 0.001, 0.01, 0.05]
TABLE_NOISE_LEVELS = [0.0, 0.01, 0.05]
BUDGETS = [3, 5, 10, 50]
ATOL = 1e-9
# Nu există seed-uri aleatoare aici: toate calculele sunt deterministe (scoruri salvate, fără simulare).

import json
import numpy as np
from sklearn.metrics import average_precision_score, f1_score
from src import scores_io as sio, alert_budget as ab, cost_ledger as cl, report as rp

RESULTS_DIR = sio.RESULTS_DIR / "result_exp8"
RESULTS_DIR.mkdir(exist_ok=True)
read_jsonl = lambda path: [json.loads(line) for line in open(path)]

## P1 – Split comun

In [ ]:
sha = sio.check_split()
y_val, y_test = sio.load_labels("val"), sio.load_labels("test")
print("sha256 split:", sha)
print("val:", len(y_val), "tranzacții,", int(y_val.sum()), "fraude | test:", len(y_test), "tranzacții,", int(y_test.sum()), "fraude")

## Reproducere din scorurile salvate

AUPRC și pragul max-F1 pe validation pentru A și B, la toate (seed, p), față de `val_runs.jsonl` din Exp. 2 și 3; AUPRC, F1 și prag pe test față de `test_runs.jsonl` din Exp. 5.

In [ ]:
worst = 0
for model, exp in [("A", "result_exp2"), ("B", "result_exp3")]:
    runs = read_jsonl(sio.RESULTS_DIR / exp / "val_runs.jsonl")
    for r in runs:
        s = sio.load_scores(model, r["seed"], r["p"], "val")
        worst = max(worst, abs(average_precision_score(y_val, s) - r["AUPRC"]), abs(ab.max_f1_threshold(y_val, s) - r["threshold"]))
    assert len(runs) == len(SEEDS) * len(NOISE_LEVELS)
print("validation: diferența maximă absolută", worst)
assert worst <= ATOL

worst, n = 0, 0
for r in read_jsonl(sio.RESULTS_DIR / "result_exp5" / "test_runs.jsonl"):
    m = r["model"]
    s_val, s_test = sio.load_scores(m, r["seed"], r["p"], "val"), sio.load_scores(m, r["seed"], r["p"], "test")
    thr = ab.max_f1_threshold(y_val, s_val)
    worst = max(worst, abs(average_precision_score(y_test, s_test) - r["AUPRC"]), abs(f1_score(y_test, s_test >= thr) - r["F1"]), abs(thr - r["threshold"]))
    n += 1
print("test:", n, "evaluări, diferența maximă absolută", worst)
assert n == 45 and worst <= ATOL

## `alert_budget` față de `pick` din Exp. 6

Scorurile B salvate de Exp. 6 pe validation trebuie să fie cele din Exp. 5, iar pragul pentru FP ≤ k trebuie să coincidă cu regula `pick` (TP maxim, apoi pragul cel mai mare).

In [ ]:
def pick_exp6(points, mask):
    cand = points[mask]
    if cand.empty:
        return None
    cand = cand[cand["TP"] == cand["TP"].max()]
    return cand["threshold"].max()

checked = 0
for seed in SEEDS:
    for p in TABLE_NOISE_LEVELS:
        with np.load(sio.RESULTS_DIR / "result_exp6" / "val_scores" / f"B_seed{seed}_p{p}.npz") as z:
            s6 = z[z.files[0]] if len(z.files) == 1 else z["val"]
        s = sio.load_scores("B", seed, p, "val")
        assert np.allclose(s6, s, atol=ATOL)
        points = ab.operating_points(y_val, s)
        for k in BUDGETS:
            assert pick_exp6(points, points["FP"] <= k) == ab.threshold_at_fp_budget(y_val, s, k)
            checked += 1
print(checked, "praguri verificate, 0 diferențe")

## Cost ledger

30 epoci, 590 exemple de antrenare, 56.962 de validare, P = 32: formula conservatoare 1 + 2P.

In [ ]:
ledger = cl.vqc_ledger()
for name, value in ledger.items():
    print(f"{name:30s} {value:>12,}")
assert ledger["antrenare"] == 1_150_500 and ledger["validare pe epoci"] == 1_708_860 and ledger["total"] == 3_315_056
print("shots nominale, 5 seeds, un model:", f"{cl.shots(5 * ledger['total']):,}")

## Tabel de referință „buget de alarme” pe validation

Pragul se alege pe validation (TP maxim cu FP ≤ k) și se citesc TP/FP pe aceeași validation, deci cifrele sunt optimiste. Modelele A, B (zgomot cuantic) și C (regresie logistică, aceleași scoruri la orice p).

In [ ]:
table = ab.alert_budget_table(y_val, y_val, lambda m, s, p: (sio.load_scores(m, s, p, "val"),) * 2,
                              ["A", "B", "C"], SEEDS, TABLE_NOISE_LEVELS, BUDGETS)
table.to_csv(RESULTS_DIR / "alert_budget_validation_per_seed.csv", index=False)
assert table["fezabil"].all()
summary = rp.summarize(table, ["TP", "FP", "recall"], group_cols=("model", "p", "k"))
summary.to_csv(RESULTS_DIR / "alert_budget_validation_summary.csv", index=False)
display_table = summary.assign(TP=lambda d: d.TP_mean.round(1).astype(str) + " ± " + d.TP_std.round(1).astype(str),
                               FP=lambda d: d.FP_mean.round(1).astype(str) + " ± " + d.FP_std.round(1).astype(str),
                               recall=lambda d: d.recall_mean.round(3).astype(str) + " ± " + d.recall_std.round(3).astype(str))
display_table.pivot(index=["model", "p"], columns="k", values="TP")

## Grafice (validation)

Același stil ca în Exp. 3–7: puncte gri = seeds, marker colorat = medie, bare = std (`ddof=1`). Modele: A (portocaliu), B (albastru), C = regresie logistică (verde). Toate cifrele sunt pe **validation** (99 de fraude), deci optimiste.

In [ ]:
import matplotlib
matplotlib.use("Agg") if not hasattr(__builtins__, "get_ipython") and "get_ipython" not in globals() else None
import matplotlib.pyplot as plt

PLOTS_DIR = RESULTS_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)
SEED_DOT_COLOR = "#8a8f98"
MODEL_STYLE = {"A": ("#eb6834", "s", "A · VQC baseline (ZZ ×2)"),
               "B": ("#2a78d6", "o", "B · VQC simplificat (ZZ ×1)"),
               "C": ("#1fa97a", "D", "C · regresie logistică")}
OFFSET = {"A": -0.1, "B": 0.0, "C": 0.1}
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False,
                     "lines.linewidth": 2})


def mean_with_seeds(ax, x, groups, model):
    color, marker, label = MODEL_STYLE[model]
    for xi, g in zip(x, groups):
        ax.scatter(np.full(len(g), xi + OFFSET[model]), g, s=18, color=SEED_DOT_COLOR, alpha=0.6, zorder=2)
    ax.errorbar(np.asarray(x) + OFFSET[model], [g.mean() for g in groups], yerr=[g.std(ddof=1) for g in groups],
                color=color, marker=marker, markersize=8, capsize=4, label=label, zorder=3)


def tp_values(model, p, k):
    return table[(table["model"] == model) & (table["p"] == p) & (table["k"] == k)].sort_values("seed")["TP"].to_numpy(dtype=float)

### 1. Fraude detectate (TP) în funcție de bugetul de alarme k, câte un panou per p

In [ ]:
fig, axes = plt.subplots(1, len(TABLE_NOISE_LEVELS), figsize=(15, 4.4), sharey=True)
xk = np.arange(len(BUDGETS))
for ax, p in zip(axes, TABLE_NOISE_LEVELS):
    for model in "ABC":
        mean_with_seeds(ax, xk, [tp_values(model, p, k) for k in BUDGETS], model)
    ax.set_xticks(xk, [f"k = {k}" for k in BUDGETS])
    ax.set_xlabel("buget de alarme false (FP ≤ k)")
    ax.set_title(f"p = {p}")
axes[0].set_ylabel("fraude detectate TP (din 99)")
axes[0].legend(loc="lower right")
fig.suptitle("TP la FP ≤ k · validation · medie ± std, 5 seeds")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "01_tp_vs_budget.png", dpi=130)
plt.show()

### 2. Cum se degradează TP odată cu zgomotul p, câte un panou per k

In [ ]:
fig, axes = plt.subplots(1, len(BUDGETS), figsize=(17, 4.2), sharey=True)
xp = np.arange(len(TABLE_NOISE_LEVELS))
for ax, k in zip(axes, BUDGETS):
    for model in "ABC":
        mean_with_seeds(ax, xp, [tp_values(model, p, k) for p in TABLE_NOISE_LEVELS], model)
    ax.set_xticks(xp, [str(p) for p in TABLE_NOISE_LEVELS])
    ax.set_xlabel("probabilitate depolarizantă p per poartă")
    ax.set_title(f"FP ≤ {k}")
axes[0].set_ylabel("fraude detectate TP (din 99)")
axes[0].legend(loc="lower left")
fig.suptitle("TP vs p la buget de alarme fix · validation · medie ± std, 5 seeds")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "02_tp_vs_p_per_budget.png", dpi=130)
plt.show()

### 3. Compromisul TP vs FP (scară log), câte o curbă per seed; liniile verticale marchează k = 3, 5, 10, 50

In [ ]:
fig, axes = plt.subplots(1, len(TABLE_NOISE_LEVELS), figsize=(15, 4.6), sharey=True)
for ax, p in zip(axes, TABLE_NOISE_LEVELS):
    for model in "ABC":
        color, _, label = MODEL_STYLE[model]
        for seed in SEEDS:
            pts = ab.operating_points(y_val, sio.load_scores(model, seed, p, "val"))
            ax.step(pts["FP"].clip(lower=0.5), pts["TP"], where="post", color=color, alpha=0.5, linewidth=1.2,
                    label=label if seed == SEEDS[0] else None)
    for k in BUDGETS:
        ax.axvline(k, color="#444444", linestyle=":", linewidth=1)
    ax.set_xscale("log")
    ax.set_xlim(0.5, 4000)
    ax.set_ylim(0, 100)
    ax.set_xlabel("alarme false FP (din 56.863 legitime, scară log)")
    ax.set_title(f"p = {p}")
axes[0].set_ylabel("fraude detectate TP (din 99)")
axes[0].legend(loc="lower right")
fig.suptitle("Compromis fraude detectate vs alarme false · validation (câte o curbă per seed)")
plt.tight_layout()
fig.savefig(PLOTS_DIR / "03_tradeoff_tp_vs_fp.png", dpi=130)
plt.show()

### 4. Cost ledger pe seed (execuții de circuit, formula 1 + 2P)

In [ ]:
parts = [(k, v) for k, v in ledger.items() if k != "total"]
fig, ax = plt.subplots(figsize=(9, 3.6))
left = 0
for (name, value), color in zip(parts, ["#2a78d6", "#eb6834", "#1fa97a", "#8a8f98"]):
    ax.barh(["VQC (A sau B)"], [value], left=left, color=color, edgecolor="white", linewidth=2, label=f"{name}: {value:,}")
    left += value
ax.set_xlabel("execuții de circuit per seed (total " + f"{ledger['total']:,}" + "; × 1024 shots)")
ax.set_title("Cost ledger per seed · identic pentru A și B · estimare nominală, nu timp de simulare")
ax.grid(axis="y", visible=False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=2, frameon=False)
plt.tight_layout()
fig.savefig(PLOTS_DIR / "04_cost_ledger.png", dpi=130, bbox_inches="tight")
plt.show()

## Independența de batch (p = 0.01, seed 0)

Scorul lui B pe un subset de 5.000 de tranzacții (un singur batch) trebuie să coincidă cu scorul aceleiași tranzacții în setul complet salvat de Exp. 5 (`atol=1e-8`), iar tranzacțiile scorate una câte una la fel. Necesar pentru afirmațiile de cost din S10 (scorarea doar a candidaților). Rulează `default.mixed`: câteva secunde.

In [ ]:
import pandas as pd
import torch
from src import circuits as circ

BATCH_SEED, BATCH_P, SUBSET_SIZE, SINGLE_SIZE, BATCH_ATOL = 0, 0.01, 5000, 200, 1e-8

df = pd.read_csv(sio.DATA_PATH)
with np.load(sio.SPLIT_PATH) as split:            # cheia "test" nu se citește
    train_idx, val_idx = split["train"], split["val"]
X, y = df.drop(columns="Class"), df["Class"].to_numpy()
X_train, y_train, X_val = X.loc[train_idx], y[train_idx], X.loc[val_idx]
del df, X                                          # testul nu rămâne în memorie

X_s, _ = circ.undersample(X_train, y_train, BATCH_SEED)
Z_val = circ.AngleEncoder(circ.N_QUBITS, BATCH_SEED).fit(X_s).transform(X_val)
weights = torch.load(sio.RESULTS_DIR / "result_exp3" / "weights" / f"seed_{BATCH_SEED}.pt", weights_only=True)["weights"]
vqc_b = circ.make_vqc(circ.FEATURE_MAP_REPS_B)
saved = sio.load_scores("B", BATCH_SEED, BATCH_P, "val")

idx = np.sort(np.random.default_rng(BATCH_SEED).choice(len(Z_val), SUBSET_SIZE, replace=False))
subset = circ.noisy_predict(Z_val[idx], weights, BATCH_P, vqc_b, chunk=SUBSET_SIZE)
single = circ.noisy_predict(Z_val[idx[:SINGLE_SIZE]], weights, BATCH_P, vqc_b, chunk=1)
d_subset, d_single = np.abs(subset - saved[idx]).max(), np.abs(single - saved[idx[:SINGLE_SIZE]]).max()
print(f"subset de {SUBSET_SIZE} într-un batch vs setul complet salvat: {d_subset:.2e}")
print(f"{SINGLE_SIZE} tranzacții una câte una vs setul complet salvat: {d_single:.2e}")
assert d_subset <= BATCH_ATOL and d_single <= BATCH_ATOL